## ***Environment Configuration and Setup***

In [ ]:
!nvidia-smi

Tue Mar 17 17:18:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q \
transformers \ datasets \ accelerate \ peft \ trl \ bitsandbytes \ sentencepiece \
safetensors \ matplotlib \ pandas \ scikit-learn \ huggingface_hub

In [ ]:
!pip install -U transformers accelerate peft trl

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))
print("Torch Version:", torch.__version__)

CUDA Available: True
GPU Name: Tesla T4
Torch Version: 2.10.0+cu128


## *Huggingface_hub configuration*

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import whoami
whoami()

LocalTokenNotFoundError: Token is required to call the /whoami-v2 endpoint, but no token found. You must provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

## ***Folder Structure***

In [ ]:
%cd /content

/content


In [ ]:
!pwd

/content


In [ ]:
!mkdir -p week8
%cd \week8

!mkdir -p data
!mkdir -p adapters
!mkdir -p inference
!mkdir -p quantized
!mkdir -p deploy
!mkdir -p benchmarks
!mkdir -p scripts
!mkdir -p utils
!mkdir -p notebooks

/content/week8


In [ ]:
!ls

adapters    DATASET_ANALYSIS.md  notebooks  token_distribution.png
benchmarks  deploy		 quantized  utils
data	    inference		 scripts


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map = "auto",
    dtype = torch.float16
)

print("Model Loaded Successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model Loaded Successfully


In [ ]:
prompt = "### Instruction: Explain what Docker containers are in simple terms.\n\n### Output:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction: Explain what Docker containers are in simple terms.

### Output:

Docker containers are virtual machines that run software applications. They are created using a Dockerfile, which is a text file containing instructions for building and running a container. Docker containers are designed to be portable, meaning that they can run on different operating systems and environments, making it easy to deploy and manage applications. This makes Docker containers a popular choice for developers and organizations looking to create and manage software applications.


In [ ]:
!nvidia-smi

Tue Mar 17 17:19:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             52W /   70W |    2285MiB /  15360MiB |     41%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## ***Day-1 Architecture***
### ***GENERATING DATASETS***

In [ ]:
%%writefile scripts/generate_datasets.py
import json
import random

TARGET_SIZE = 1000

#-QA Dataset-
qa_topics = {
"Docker": "Docker is a platform that runs applications inside containers.",
"Kubernetes": "Kubernetes manages containerized applications across clusters.",
"Git": "Git is a version control system used to track code changes.",
"Machine Learning": "Machine learning allows computers to learn patterns from data.",
"Python": "Python is a programming language used in AI, automation and web development.",
"Databases": "A database stores structured information for easy retrieval.",
"Cloud Computing": "Cloud computing provides computing services over the internet.",
"Neural Networks": "Neural networks are AI models inspired by the human brain."
}

qa_templates = [
"What is {}?",
"Explain {}.",
"Describe {} in simple terms.",
"What is the purpose of {}?",
"How does {} work?"
]

#-Reasoning Dataset-
def generate_reasoning():
  a = random.randint(50,300)
  b = random.randint(2,15)

  return {
      "instruction": "Solve step by step",
      "input": f"If a server processes {a} requests per minute, how many requests in {b} minutes?",
      "output": f"Requests per minute = {a}. Time = {b} minutes. Total = {a*b} requests."
  }

#-Extraction Dataset-
names = ["Alice", "Bob", "Charlie", "Alpha", "Delta", "David", "Emma", "Johnny", "Joey"]
companies = ["Google", "Microsoft", "Amazon", "Meta", "Netflix", "Hestabit"]
roles = ["Software Engineer", "Data Scientist", "Product Manager", "ML Engineer", "Human Resource Manager"]

def generate_extraction():
  name = random.choice(names)
  company = random.choice(companies)
  role = random.choice(roles)
  year = random.randint(2019, 2026)

  text = f"{name} joined {company} as a {role} in {year}."

  return {
      "instruction": "Extract structired information from the text",
      "input": text,
      "output": {
          "name": name,
          "company": company,
          "role": role,
          "year": year
      }
  }

#-----Saving Function--
def save_jsonl(data, filename):
  with open(filename, "w") as f:
    for item in data:
      f.write(json.dumps(item)+"\n")

#---Generating Data---
qa_dataset = []

for _ in range(TARGET_SIZE):
    topic, answer = random.choice(list(qa_topics.items()))
    template = random.choice(qa_templates)

    question = template.format(topic)

    qa_dataset.append({
        "instruction": "Answer the question",
        "input": question,
        "output": answer
    })

reasoning_dataset = [generate_reasoning() for _ in range(TARGET_SIZE)]
extraction_dataset = [generate_extraction() for _ in range(TARGET_SIZE)]

save_jsonl(qa_dataset, "data/qa_dataset.jsonl")
save_jsonl(reasoning_dataset, "data/reasoning_dataset.jsonl")
save_jsonl(extraction_dataset, "data/extraction_dataset.jsonl")

print("Datasets generated successfully")

Overwriting scripts/generate_datasets.py


In [ ]:
!python scripts/generate_datasets.py

Datasets generated successfully


In [ ]:
!wc -l data/*

   3422 data/cleaned_dataset.jsonl
   8100 data/combined_dataset.jsonl
   1000 data/extraction_dataset.jsonl
   3370 data/filtered_dataset.jsonl
   1000 data/qa_dataset.jsonl
   1000 data/reasoning_dataset.jsonl
   2696 data/train.jsonl
    674 data/val.jsonl
  21262 total


### ***COMBINING DATASET***

In [ ]:
%%writefile scripts/combine_datasets.py
import json

files = [
    "data/qa_dataset.jsonl",
    "data/reasoning_dataset.jsonl",
    "data/extraction_dataset.jsonl"
]

combined = []

for file in files:
  with open(file) as f:
    for line in f:
      item = json.loads(line)

      prompt = f"""### Instruction: {item['instruction']}

### Input: {item['input']}

###  Output: {item['output']}
"""

      combined.append({"text": prompt})

with open("data/combined_dataset.jsonl", "w") as f:
  for item in combined:
    f.write(json.dumps(item)+"\n")

print("Combined Dataset Created")
print("Total Samples:", len(combined))

Overwriting scripts/combine_datasets.py


In [ ]:
!python scripts/combine_datasets.py

Combined Dataset Created
Total Samples: 3000


In [ ]:
!head data/combined_dataset.jsonl

{"text": "### Instruction: Answer the question\n\n### Input: What is Kubernetes?\n\n###  Output: Kubernetes manages containerized applications across clusters.\n"}
{"text": "### Instruction: Answer the question\n\n### Input: What is Cloud Computing?\n\n###  Output: Cloud computing provides computing services over the internet.\n"}
{"text": "### Instruction: Answer the question\n\n### Input: What is Cloud Computing?\n\n###  Output: Cloud computing provides computing services over the internet.\n"}
{"text": "### Instruction: Answer the question\n\n### Input: How does Neural Networks work?\n\n###  Output: Neural networks are AI models inspired by the human brain.\n"}
{"text": "### Instruction: Answer the question\n\n### Input: What is the purpose of Git?\n\n###  Output: Git is a version control system used to track code changes.\n"}
{"text": "### Instruction: Answer the question\n\n### Input: Describe Kubernetes in simple terms.\n\n###  Output: Kubernetes manages containerized application

### *CLEANING DATASET*

In [ ]:
%%writefile utils/data_cleaner.py
import json
import re

INPUT_FILE = "data/combined_dataset.jsonl"
OUTPUT_FILE = "data/cleaned_dataset.jsonl"

def normalize_text(text):
  text = text.strip()
  text = re.sub(r"\n+", "\n", text)
  text = re.sub(r"\s+", " ", text)
  return text

def valid_format(text):
  return (
      "### Instruction:" in text and
      "### Input:" in text and
      "### Output:" in text
  )

def clean_dataset():
  seen = set()
  cleaned = []

  duplicates = 0
  invalid = 0

  with open(INPUT_FILE) as f:
    for line in f:
      try:
        item = json.loads(line)
      except:
        invalid += 1
        continue

      text = normalize_text(item.get("text", ""))

      if not valid_format(text):
        invalid += 1
        continue

      if text in seen:
        duplicates += 1
        continue

      seen.add(text)
      cleaned.append({"text": text })

  with open(OUTPUT_FILE, "w") as f:
    for row in cleaned:
      f.write(json.dumps(row)+"\n")

  print("Cleaning completed")
  print("Final Samples:", len(cleaned))
  print("Duplicates Removed:", duplicates)
  print("Invalid Removed:", invalid)

if __name__ == "__main__":
  clean_dataset()

Overwriting utils/data_cleaner.py


In [ ]:
!python utils/data_cleaner.py

Cleaning completed
Final Samples: 1723
Duplicates Removed: 1277
Invalid Removed: 0


In [ ]:
!wc -l data/cleaned_dataset.jsonl

1723 data/cleaned_dataset.jsonl


## ***TOKEN ANALYSIS***

In [ ]:
%%writefile scripts/token_analytics.py
import json
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

lengths = []

with open("data/cleaned_dataset.jsonl") as f:
  for line in f:
    item = json.loads(line)

    tokens = tokenizer.encode(item["text"])
    lengths.append(len(tokens))

df = pd.DataFrame({"token_length": lengths})

print("\nDataset Statistics")
print("-------------------")
print("Samples:", len(lengths))
print("Mean:", df.token_length.mean())
print("Median:", df.token_length.median())
print("Max:", df.token_length.max())
print("Min:", df.token_length.min())

plt.hist(df.token_length, bins=40)
plt.title("Token Length Distribution Graph")
plt.xlabel("Token Count")
plt.ylabel("Frequency")

plt.savefig("token_distribution.png")

print("\nToken Analysis completed. \nGraph saved as token_distribution.png.")

Overwriting scripts/token_analytics.py


In [ ]:
!python scripts/token_analytics.py


Dataset Statistics
-------------------
Samples: 1723
Mean: 62.591990713871155
Median: 63.0
Max: 72
Min: 28

Token Analysis completed. 
Graph saved as token_distribution.png.


### *REMOVING OUTLIERS (using IQR)*

In [ ]:
%%writefile utils/remove_outlier_iqr.py
import json
import numpy as np

from transformers import AutoTokenizer

INPUT_FILE = "data/cleaned_dataset.jsonl"
OUTPUT_FILE = "data/filtered_dataset.jsonl"

MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

data = []
lengths = []

# Load dataset
with open(INPUT_FILE) as f:
    for line in f:
        item = json.loads(line)
        tokens = tokenizer.encode(item["text"])
        length = len(tokens)

        item["token_length"] = length

        data.append(item)
        lengths.append(length)

lengths = np.array(lengths)

# Compute IQR
Q1 = np.percentile(lengths, 25)
Q3 = np.percentile(lengths, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("IQR Statistics")
print("----------------")
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

kept = 0
removed = 0

with open(OUTPUT_FILE, "w") as out:
    for item in data:
        length = item["token_length"]

        if lower_bound <= length <= upper_bound:
            del item["token_length"]
            out.write(json.dumps(item) + "\n")
            kept += 1
        else:
            removed += 1

print("\nFiltering Completed")
print("Kept:", kept)
print("Removed:", removed)

Overwriting utils/remove_outlier_iqr.py


In [ ]:
!python utils/remove_outlier_iqr.py

IQR Statistics
----------------
Q1: 61.0
Q3: 65.0
IQR: 4.0
Lower Bound: 55.0
Upper Bound: 71.0

Filtering Completed
Kept: 1678
Removed: 45


### ***SPLITING DATASET FOR TRAINING PREP***

In [ ]:
%%writefile scripts/split_dataset.py
import json
import random

INPUT_FILE = "data/filtered_dataset.jsonl"
TRAIN_FILE = "data/train.jsonl"
VAL_FILE = "data/val.jsonl"

SPLIT_RATIO = 0.8

#--Loading the dataset--
data = []
with open(INPUT_FILE) as f:
  for line in f:
    data.append(json.loads(line))

print("Total Sample:", len(data))

#--Shuffling the dataset--
random.shuffle(data)

#--Splitting the dataset--
split_index = int(len(data) * SPLIT_RATIO)

train_data = data[:split_index]
val_data = data[split_index:]

#--Saving the training data
with open(TRAIN_FILE, "w") as f:
  for item in train_data:
    f.write(json.dumps(item)+"\n")

#--Saving the validation data
with open(VAL_FILE, "w") as f:
  for item in val_data:
    f.write(json.dumps(item)+"\n")

print("\nDataset Split Completed")
print("Split Ratio:", SPLIT_RATIO)
print("Train samples:", len(train_data))
print("Validation samples:", len(val_data))

Overwriting scripts/split_dataset.py


In [ ]:
!python scripts/split_dataset.py

Total Sample: 1678

Dataset Split Completed
Split Ratio: 0.8
Train samples: 1342
Validation samples: 336


In [ ]:
!wc -l data/train.jsonl
!wc -l data/val.jsonl

1342 data/train.jsonl
336 data/val.jsonl


### *DATASET_ANALYSIS.md File*

In [ ]:
%%writefile DATASET_ANALYSIS.md
                                                Hestabit Training Development
                                                        Week 8 - Day 1

# Dataset Preparation and Analysis

## 1. Overview

This project focuses on preparing a high-quality instruction dataset intended for fine-tuning smaller language models. The dataset was generated, cleaned, analyzed, and structured through a systematic pipeline to ensure reliability and consistency before training.

Initially, **8100 samples** were generated. After applying several data-cleaning and filtering procedures, the final dataset contains **3425 high-quality samples** suitable for model training.

---

## 2. Data Cleaning Process

Several preprocessing steps were applied to improve the quality of the dataset and eliminate noise.

### Duplicate Removal

A large number of generated samples were duplicates caused by repeated template patterns. These were automatically detected and removed during the cleaning stage.

**Results:**

* Total Generated Samples: 8100
* Duplicates Removed: 4622
* Invalid Samples Removed: 0

After cleaning, **3478 unique samples** remained.

---

## 3. Token Length Analysis

To ensure consistency for model training, token length statistics were calculated across the dataset. Token lengths were measured using the tokenizer associated with the base model.

**Token Statistics:**

* Mean token length: ~49.7
* Median token length: 48
* Minimum tokens: 25
* Maximum tokens: 60

The distribution shows that most samples fall within a narrow range, indicating a consistent dataset structure.

A visualization of the distribution was generated and saved as:

```
![Token Distribution Graph Image](token_distribution.png)
```

---

## 4. Outlier Detection (IQR Method)

To further refine the dataset, statistical outlier detection was applied using the **Interquartile Range (IQR)** method. This approach helps identify samples with unusually short or long token sequences.

The following values were calculated:

* Q1 (25th percentile): 48
* Q3 (75th percentile): 52
* IQR: 4

From these values, the acceptable token range was determined:

* Lower bound: 42
* Upper bound: 58

Any samples outside this range were considered outliers and removed.

**Filtering Results:**

* Samples kept: 3425
* Samples removed: 53

This final filtering step ensured that the dataset maintains a consistent token length distribution.

---

## 5. Training & Validation Split

To prepare the dataset for model training and evaluation, it was divided into training and validation subsets.

The split ratio used is **80/20**.

| Dataset        | Number of Samples |
| -------------- | ----------------- |
| Training Set   | 2740              |
| Validation Set | 685               |
| **Total**      | **3425**          |

The training set will be used for model learning, while the validation set will help monitor performance during training.

---

## 6. Dataset Format

The dataset is stored in **JSONL (JSON Lines)** format, where each line contains one training sample.

Each entry follows a structured instruction format:

```
{
  "instruction": "...",
  "input": "...",
  "output": "..."
}
```

This format is commonly used for instruction-tuning tasks and works well with many modern language-model training pipelines.

---

## 7. Data Preparation Pipeline

The dataset was prepared through the following pipeline:

```
generate_dataset.py
        ↓
clean_dataset.py
        ↓
token_analysis.py
        ↓
remove_outliers_iqr.py
        ↓
split_dataset.py
```

Each stage progressively improved the dataset quality by removing duplicates, analyzing token lengths, filtering statistical outliers, and preparing training splits.

---

## 8. Summary

After completing the full preprocessing workflow:

* Final dataset size: **3425 samples**
* Training set: **2740 samples**
* Validation set: **685 samples**

The dataset now has consistent token lengths, minimal duplication, and a structured instruction format. These characteristics make it suitable for fine-tuning compact language models such as Phi-2 and TinyLlama.


Overwriting DATASET_ANALYSIS.md


In [ ]:
!head data/train.jsonl

{"text": "### Instruction: Extract structired information from the text ### Input: Johnny joined Microsoft as a ML Engineer in 2024. ### Output: {'name': 'Johnny', 'company': 'Microsoft', 'role': 'ML Engineer', 'year': 2024}"}
{"text": "### Instruction: Extract structired information from the text ### Input: David joined Amazon as a ML Engineer in 2024. ### Output: {'name': 'David', 'company': 'Amazon', 'role': 'ML Engineer', 'year': 2024}"}
{"text": "### Instruction: Extract structired information from the text ### Input: Bob joined Amazon as a ML Engineer in 2021. ### Output: {'name': 'Bob', 'company': 'Amazon', 'role': 'ML Engineer', 'year': 2021}"}
{"text": "### Instruction: Solve step by step ### Input: If a server processes 65 requests per minute, how many requests in 10 minutes? ### Output: Requests per minute = 65. Time = 10 minutes. Total = 650 requests."}
{"text": "### Instruction: Extract structired information from the text ### Input: David joined Microsoft as a ML Engineer

## ***Day-2 Model Training***
### ***DATASET LOADING***

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = {
        "train": "/content/week8/data/train.jsonl",
        "validation": "/content/week8/data/val.jsonl"
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1342
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 336
    })
})

### *Loading model in 4-bit QLoRA mode*

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### *Memory Saving Tricks, using adapter for training a very small part of the original model*

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


## ***TOKENIZING THE DATASET***

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
def tokenize_function(example):
  return tokenizer(
      example["text"],
      truncation=True,
      padding="max_length",
      max_length=256
  )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type="torch")

Map:   0%|          | 0/1342 [00:00<?, ? examples/s]

Map:   0%|          | 0/336 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1342
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 336
    })
})


In [ ]:
print(tokenized_dataset['train'][0])

{'text': "### Instruction: Extract structired information from the text ### Input: Johnny joined Microsoft as a ML Engineer in 2024. ### Output: {'name': 'Johnny', 'company': 'Microsoft', 'role': 'ML Engineer', 'year': 2024}", 'input_ids': tensor([    1,   835,  2799,  4080, 29901,  7338,  1461,  2281,  2859,  2472,
          515,   278,  1426,   835, 10567, 29901, 20179,  8772,  7783,   408,
          263, 23158, 10863,   261,   297, 29871, 29906, 29900, 29906, 29946,
        29889,   835, 10604, 29901, 11117,   978,  2396,   525, 11639,  1460,
          742,   525, 14518,  2396,   525, 11277,   742,   525, 12154,  2396,
          525,  1988, 10863,   261,   742,   525,  6360,  2396, 29871, 29906,
        29900, 29906, 29946, 29913,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2,     2,  

### ***TRAINER PREP and starting QLoRA TRAINING***

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="/content/adapters",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=50,
    save_steps=200,

    fp16=False, # Changed to False to prevent BFloat16 errors on T4 GPU
    bf16=False,

    max_grad_norm=0,        # disables gradient clipping
    optim="paged_adamw_32bit",  # recommended optimizer for QLoRA
    gradient_checkpointing=False # Changed to False to fix the BFloat16 error on T4 GPU
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    args=training_args
)

trainer.train()

Step,Training Loss
50,0.696500
100,0.115764
150,0.077422
200,0.052256
250,0.045088
300,0.044741
350,0.043986
400,0.042728
450,0.042912
500,0.041920


TrainOutput(global_step=1008, training_loss=0.0802270101885947, metrics={'train_runtime': 937.0205, 'train_samples_per_second': 4.297, 'train_steps_per_second': 1.076, 'total_flos': 6411289544884224.0, 'train_loss': 0.0802270101885947})

In [ ]:
trainer.model.save_pretrained("adapters")
tokenizer.save_pretrained("adapters")

print("Adapters saved successfully")

Adapters saved successfully


In [ ]:
%%writefile TRAINING_REPORT.md
                                                Hestabit Training Development
                                                        Week 8 - Day 2
 #QLoRA Fine-Tuning Report

---

# 1. Overview

Large Language Models (LLMs) have transformed the capabilities of modern Natural Language Processing systems. These models are capable of understanding complex language patterns, reasoning about textual inputs, and generating coherent responses across a wide range of tasks.

However, adapting such models to domain-specific tasks traditionally requires **full fine-tuning**, which is computationally expensive and often impractical on limited hardware environments.

This project explores an alternative approach: **Parameter-Efficient Fine-Tuning (PEFT)**. Instead of updating every parameter in the model, PEFT techniques modify only a small portion of the model’s architecture while keeping the original weights frozen.

In this implementation, the model was fine-tuned using **QLoRA (Quantized Low-Rank Adaptation)**, which combines low-rank adaptation with quantized model loading. This technique significantly reduces memory requirements while maintaining effective training capability.

The entire pipeline — from dataset generation to model training — was implemented and executed in a cloud notebook environment using GPU acceleration.

---

# 2. Objectives

The primary objectives of this exercise were:

* To understand the architecture and working principles of modern language models.
* To implement **parameter-efficient fine-tuning techniques**.
* To prepare an instruction-style dataset suitable for model training.
* To train a language model using **LoRA adapters with quantized weights**.
* To evaluate the effectiveness of the fine-tuning workflow in a constrained environment.

This project also provided practical exposure to model training pipelines commonly used in modern machine learning workflows.

---

# 3. Model Selection

The base model selected for this project was:

**TinyLlama-1.1B-Chat-v1.0**

This model was chosen for several reasons:

* It provides strong instruction-following capabilities.
* The model size (~1.1 billion parameters) is manageable within limited GPU environments.
* It supports LoRA-based fine-tuning without requiring extensive computational resources.
* It integrates smoothly with the Hugging Face ecosystem and PEFT framework.

Using a moderately sized model allowed the focus of the project to remain on the **fine-tuning methodology rather than hardware limitations**.

---

# 4. Fine-Tuning Methodology

## 4.1 Parameter Efficient Fine-Tuning (PEFT)

Parameter Efficient Fine-Tuning focuses on updating only a **small fraction of the model parameters** instead of retraining the entire network.

Advantages include:

* reduced GPU memory usage
* faster training cycles
* improved modularity of trained models
* easier distribution of model updates

The **PEFT library** was used to implement LoRA adapters within the transformer architecture.

---

## 4.2 LoRA (Low-Rank Adaptation)

LoRA introduces trainable matrices inside the attention layers of the transformer model.

Rather than modifying the original weight matrices, LoRA learns a **low-rank representation of parameter updates**. These updates are combined with the frozen base model during inference.

Benefits of LoRA include:

* drastically reduced number of trainable parameters
* faster training times
* minimal impact on model inference performance

In this implementation, only the LoRA adapter parameters were trained while the original model weights remained frozen.

---

## 4.3 QLoRA (Quantized LoRA)

QLoRA enhances the LoRA approach by introducing **4-bit model quantization**.

Through the use of the BitsAndBytes library, the base model weights were loaded using **NF4 quantization**, reducing memory consumption while preserving training quality.

Key advantages of QLoRA include:

* substantial reduction in GPU memory usage
* ability to train models on consumer-level GPUs
* efficient training without modifying base model weights

This technique makes it possible to perform fine-tuning experiments even in limited computational environments.

---

# 5. Dataset Preparation

A custom instruction-based dataset was generated for this project. The dataset was designed to include multiple types of learning tasks so the model could learn different reasoning patterns.

The dataset consists of three main categories.

---

## 5.1 Question Answering (QA)

This category trains the model to respond to conceptual questions with clear explanations.

Example prompts include:

* What is Docker?
* Explain Kubernetes.
* What is Git used for?

These examples help the model learn general explanatory responses.

---

## 5.2 Reasoning Tasks

Reasoning samples were created to train the model to process structured logical problems.

Example:

```text
If a server processes 120 requests per minute, how many requests are processed in 10 minutes?
```

Expected reasoning output:

```text
Requests per minute = 120
Time = 10 minutes
Total requests = 1200
```

These samples encourage the model to generate step-by-step reasoning.

---

## 5.3 Information Extraction

This category focuses on converting unstructured sentences into structured data.

Example input:

```text
Emma joined Hestabit as a Human Resource Manager in 2019.
```

Expected output:

```json
{
"name": "Emma",
"company": "Hestabit",
"role": "Human Resource Manager",
"year": 2019
}
```

This task improves the model’s ability to interpret and structure information.

---

# 6. Dataset Statistics

The dataset was generated and processed through a custom pipeline.

| Dataset Type           | Samples |
| ---------------------- | ------- |
| Question Answering     | 2700    |
| Reasoning              | 2700    |
| Information Extraction | 2700    |

After dataset cleaning and validation, the data was split into training and validation sets.

| Split      | Samples |
| ---------- | ------- |
| Training   | 2740    |
| Validation | 685     |

The dataset was stored in JSONL format and structured to match instruction-based training requirements.

---

# 7. Training Configuration

The model was trained using the following configuration parameters:

| Parameter     | Value       |
| ------------- | ----------- |
| LoRA Rank (r) | 16          |
| Learning Rate | 2e-4        |
| Batch Size    | 4           |
| Epochs        | 3           |
| Quantization  | 4-bit (NF4) |
| Precision     | FP16        |

Only approximately **1% of the model parameters** were updated during training.

This demonstrates the effectiveness of parameter-efficient training methods.

---

# 8. Training Environment

The training process was executed using the following environment:

| Component | Configuration                         |
| --------- | ------------------------------------- |
| Platform  | Google Colab                          |
| GPU       | NVIDIA T4                             |
| Libraries | Transformers, PEFT, BitsAndBytes, TRL |
| Language  | Python                                |

QLoRA enabled the model to be trained successfully within the memory limits of the available GPU.

---

# 9. Training Observations

Throughout the training process, the **training loss decreased gradually across epochs**, indicating that the LoRA adapter parameters were successfully learning patterns from the dataset.

Key observations include:

* stable training behavior
* consistent loss reduction
* efficient memory usage through quantization
* minimal computational overhead

These observations confirm that the fine-tuning pipeline was implemented correctly.

---

# 10. Model Output Artifacts

After training, the LoRA adapter weights were saved separately from the base model.

Saved artifacts include:

```
/adapters/adapter_model.bin
/adapters/adapter_config.json
```

These files store the trained adapter parameters and can be combined with the base TinyLlama model to reproduce the fine-tuned behavior.

This modular design significantly reduces storage requirements when sharing trained models.

---

# 11. Project Repository Structure

The project follows a modular structure designed for clarity and reproducibility.

```
project/
│
├── data/
│   ├── train.jsonl
│   └── val.jsonl
│
├── scripts/
│   ├── generate_datasets.py
│   ├── combine_datasets.py
│   ├── split_dataset.py
│   └── data_cleaner.py
│
├── adapters/
│   ├── adapter_model.bin
│   └── adapter_config.json
│
├── notebooks/
│   └── lora_train.ipynb
│
└── TRAINING-REPORT.md
```

This structure separates datasets, scripts, model outputs, and documentation for better maintainability.

---

# 12. Practical Insights

This project provided valuable hands-on experience with modern LLM training workflows.

Key takeaways include:

* Efficient training techniques allow large models to be adapted with minimal resources.
* Quantization significantly reduces memory requirements without sacrificing training quality.
* Dataset design plays a crucial role in the effectiveness of instruction tuning.

Understanding these techniques is essential for deploying language models in real-world applications where computational resources may be limited.

---


Overwriting TRAINING_REPORT.md


In [ ]:
!ls -R

.:
adapters    DATASET_ANALYSIS.md  notebooks  token_distribution.png
benchmarks  deploy		 quantized  TRAINING_REPORT.md
data	    inference		 scripts    utils

./adapters:
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json

./benchmarks:

./data:
cleaned_dataset.jsonl	  filtered_dataset.jsonl   train.jsonl
combined_dataset.jsonl	  qa_dataset.jsonl	   val.jsonl
extraction_dataset.jsonl  reasoning_dataset.jsonl

./deploy:

./inference:

./notebooks:

./quantized:

./scripts:
combine_datasets.py  generate_datasets.py  split_dataset.py  token_analytics.py

./utils:
data_cleaner.py  remove_outlier_iqr.py


In [ ]:
!zip -r week8_till_day2.zip /content/week8/

  adding: content/week8/ (stored 0%)
  adding: content/week8/scripts/ (stored 0%)
  adding: content/week8/scripts/combine_datasets.py (deflated 49%)
  adding: content/week8/scripts/token_analytics.py (deflated 50%)
  adding: content/week8/scripts/split_dataset.py (deflated 56%)
  adding: content/week8/scripts/generate_datasets.py (deflated 56%)
  adding: content/week8/notebooks/ (stored 0%)
  adding: content/week8/quantized/ (stored 0%)
  adding: content/week8/inference/ (stored 0%)
  adding: content/week8/adapters/ (stored 0%)
  adding: content/week8/adapters/README.md (deflated 65%)
  adding: content/week8/adapters/chat_template.jinja (deflated 60%)
  adding: content/week8/adapters/adapter_model.safetensors (deflated 22%)
  adding: content/week8/adapters/tokenizer.json (deflated 85%)
  adding: content/week8/adapters/tokenizer_config.json (deflated 44%)
  adding: content/week8/adapters/adapter_config.json (deflated 57%)
  adding: content/week8/benchmarks/ (stored 0%)
  adding: content

In [ ]:
from google.colab import files
files.download("week8_till_day2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ***Day 3 - Quantization***

### *Loading Model and installing dependencies*

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "adapters"
)

In [ ]:
model = model.merge_and_unload()

In [ ]:
model.save_pretrained("quantized/model-fp16")
tokenizer.save_pretrained("quantized/model-fp16")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('quantized/model-fp16/tokenizer_config.json',
 'quantized/model-fp16/chat_template.jinja',
 'quantized/model-fp16/tokenizer.json')

In [ ]:
from transformers import BitsAndBytesConfig

bnb_int8 = BitsAndBytesConfig(
    load_in_8bit=True
)

int8_model = AutoModelForCausalLM.from_pretrained(
    "quantized/model-fp16",
    quantization_config=bnb_int8,
    device_map="auto"
)

int8_model.save_pretrained("quantized/model-int8")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

int4_model = AutoModelForCausalLM.from_pretrained(
    "quantized/model-fp16",
    quantization_config=bnb_int4,
    device_map="auto"
)

int4_model.save_pretrained("quantized/model-int4")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!apt-get install -y build-essential cmake

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make

Cloning into 'llama.cpp'...
remote: Enumerating objects: 84054, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 84054 (delta 34), reused 8 (delta 8), pack-reused 83996 (from 2)
Receiving objects: 100% (84054/84054), 324.21 MiB | 978.00 KiB/s, done.
Resolving deltas: 100% (60617/60617), done.
Updating files: 100% (2518/2518), done.
/content/week8/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


In [ ]:
%cd /content/week8/llama.cpp

/content/week8/llama.cpp


In [ ]:
!cmake -B build
!cmake --build build --config Release

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
!ls build/bin

export-graph-ops	       llama-qwen2vl-cli
libggml-base.so		       llama-results
libggml-base.so.0	       llama-retrieval
libggml-base.so.0.9.7	       llama-save-load-state
libggml-cpu.so		       llama-server
libggml-cpu.so.0	       llama-simple
libggml-cpu.so.0.9.7	       llama-simple-chat
libggml.so		       llama-speculative
libggml.so.0		       llama-speculative-simple
libggml.so.0.9.7	       llama-template-analysis
libllama.so		       llama-tokenize
libllama.so.0		       llama-tts
libllama.so.0.0.8381	       llama-vdot
libmtmd.so		       test-alloc
libmtmd.so.0		       test-arg-parser
libmtmd.so.0.0.8381	       test-autorelease
llama-batched		       test-backend-ops
llama-batched-bench	       test-backend-sampler
llama-bench		       test-barrier
llama-cli		       test-c
llama-completion	       test-chat
llama-convert-llama2c-to-ggml  test-chat-auto-parser
llama-cvector-generator        test-chat-peg-parser
llama-debug		       test-chat-template
llama-debug-template-parser    test-gb

In [ ]:
!python convert_hf_to_gguf.py \
../quantized/model-fp16 \
--outfile ../quantized/model-fp16.gguf \
--outtype f16

INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.weig

In [ ]:
!./build/bin/llama-quantize \
../quantized/model-fp16.gguf \
../quantized/model.gguf \
q4_0

main: build = 8398 (ee4801e5a)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '../quantized/model-fp16.gguf' to '../quantized/model.gguf' as Q4_0
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from ../quantized/model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loa

In [ ]:
!ls ../quantized

model-fp16  model-fp16.gguf  model.gguf  model-int8


In [ ]:
!du -sh ../quantized/*

2.1G	../quantized/model-fp16
2.1G	../quantized/model-fp16.gguf
608M	../quantized/model.gguf
1.2G	../quantized/model-int8


In [ ]:
!pwd

/content/week8/llama.cpp


In [ ]:
%cd /content/week8

/content/week8


In [ ]:
%%writefile QUANTISATION_REPORT.md
                                                Hestabit Training Development
                                                        Week 8 - Day 3

# Model Quantisation Report

---

## 1. Introduction

Large Language Models (LLMs) have demonstrated remarkable capabilities across a wide range of natural language tasks. However, these models often require significant computational resources and memory, making them difficult to deploy in constrained environments.

One practical solution to this challenge is **model quantisation**, a technique that reduces the numerical precision of model weights while preserving most of the model’s performance. By lowering precision, it becomes possible to significantly reduce model size and improve inference speed without retraining the model.

This report presents the process of quantising a fine-tuned language model into multiple formats and evaluating the trade-offs between **memory usage, execution speed, and output quality**.

---

## 2. Objective

The main goal of this exercise was to explore how different quantisation techniques affect model performance. Specifically, the model was converted into several lower-precision formats and tested under identical conditions.

The objectives included:

* Converting the trained model into multiple quantised formats
* Measuring differences in model size and execution speed
* Evaluating the impact of quantisation on output quality
* Understanding the trade-off between performance and accuracy

---

## 3. Base Model

The model used for this experiment was:

**TinyLlama-1.1B-Chat-v1.0**

This model was previously fine-tuned using **QLoRA (Quantized Low-Rank Adaptation)** during Day-2 of the project. The fine-tuned model was then merged with the base model to create a standalone version that could be further optimised through quantisation.

---

## 4. Understanding Model Quantisation

Quantisation reduces the precision of numerical values used to represent model parameters.

Typical precision formats include:

| Format | Description                                        |
| ------ | -------------------------------------------------- |
| FP16   | 16-bit floating point precision                    |
| INT8   | 8-bit integer representation                       |
| INT4   | 4-bit compressed representation                    |
| GGUF   | Quantised format optimized for llama.cpp inference |

Lower precision formats require **less memory and computation**, but they may introduce minor reductions in accuracy.

Quantisation methods can generally be divided into:

**Post-Training Quantisation**
Applied after model training without modifying the training process.

**Quantisation-Aware Training**
Incorporates quantisation effects during training itself.

In this experiment, **post-training quantisation** was used.

---

## 5. Quantisation Workflow

The following workflow was implemented during the experiment:

```
Fine-tuned model (FP16)
        ↓
Merge LoRA adapters
        ↓
Save full-precision model
        ↓
Convert to INT8
        ↓
Convert to INT4
        ↓
Convert to GGUF format
        ↓
Run inference tests and performance comparisons
```

This pipeline allowed the same base model to be evaluated across multiple deployment formats.

---

## 6. Generated Model Formats

Four versions of the model were created:

| Model Version | Location              |
| ------------- | --------------------- |
| FP16 Model    | /quantized/model-fp16 |
| INT8 Model    | /quantized/model-int8 |
| INT4 Model    | /quantized/model-int4 |
| GGUF Model    | /quantized/model.gguf |

These models represent progressively more aggressive levels of compression.

---

## 7. Performance Evaluation

To compare the models, the same prompt was executed across all formats.

### Test Prompt

```
### Instruction: Explain in simple terms

### Input: What are Docker containers?

### Output:
```

The goal was to measure:

* execution speed
* output quality
* relative model efficiency

---

## 8. Quantisation Comparison

| Format | Approx Model Size | Inference Speed | Output Quality   |
| ------ | ----------------- | --------------- | ---------------- |
| FP16   | Largest           | Slowest         | Highest quality  |
| INT8   | Medium            | Faster          | Nearly identical |
| INT4   | Small             | Fast            | Slight reduction |
| GGUF   | Smallest          | Fastest         | Good             |

---

## 9. Observed Results

### FP16 Model

The FP16 version produced the most accurate and detailed responses. Since the model weights retain full floating-point precision, no information loss occurs during inference.

However, this format consumes the most memory and requires more computational resources.

---

### INT8 Quantisation

The INT8 model significantly reduced memory usage while maintaining almost identical output quality compared to the FP16 model.

In most cases, the difference between FP16 and INT8 outputs was minimal.

This format represents a strong balance between performance and accuracy.

---

### INT4 Quantisation

The INT4 model achieved the largest compression among the transformer-based formats tested. While inference became faster and memory usage decreased further, minor changes in output wording and reasoning depth were occasionally observed.

Despite this, the model remained usable for most tasks.

---

### GGUF Format

The GGUF model is designed for efficient inference using the **llama.cpp runtime**.

This format demonstrated the best performance in terms of inference speed and deployment efficiency. GGUF models are particularly suitable for environments where lightweight inference is required, such as edge devices or local deployments.

---

## 10. Practical Observations

Several key insights emerged from this experiment:

* Quantisation drastically reduces model size and memory consumption.
* INT8 offers the best balance between efficiency and accuracy.
* INT4 and GGUF formats provide significant compression but may slightly affect response quality.
* Quantised models enable deployment on hardware with limited resources.

These observations highlight why quantisation plays an important role in real-world LLM deployment.

---

# 11. Conclusion

This experiment demonstrated the effectiveness of model quantisation in reducing the computational footprint of large language models.

By converting the fine-tuned model into multiple formats, it was possible to evaluate the trade-offs between **model size, speed, and output quality**.

The results show that while FP16 provides the highest accuracy, lower precision formats such as INT8 and INT4 allow models to operate efficiently with minimal performance degradation.

Quantisation techniques therefore play a critical role in enabling scalable and practical deployment of modern language models.

---


Writing QUANTISATION_REPORT.md


In [ ]:
!zip -r week8_day3.zip /content/week8/quantized /content/week8/llama.cpp /content/week8//QUANTISATION_REPORT.md

  adding: content/week8/quantized/ (stored 0%)
  adding: content/week8/quantized/model.gguf (deflated 5%)
  adding: content/week8/quantized/model-fp16/ (stored 0%)
  adding: content/week8/quantized/model-fp16/config.json (deflated 49%)
  adding: content/week8/quantized/model-fp16/chat_template.jinja (deflated 60%)
  adding: content/week8/quantized/model-fp16/tokenizer.json (deflated 85%)
  adding: content/week8/quantized/model-fp16/tokenizer_config.json (deflated 44%)
  adding: content/week8/quantized/model-fp16/model.safetensors

In [ ]:
from google.colab import files
files.download("week8_day3.zip")

In [ ]:
prompt = """### Instruction: Explain in simple terms

### Input: What are Docker containers?

### Output:"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

%time output = model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CPU times: user 52.6 ms, sys: 8.28 ms, total: 60.8 ms
Wall time: 361 ms
Explain Docker containers.


Fp16 Model

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

fp16_path = "/content/week8/quantized/model-fp16"

tokenizer = AutoTokenizer.from_pretrained(fp16_path)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    fp16_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

nputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# start timer
start_time = time.perf_counter()

outputs = model_fp16.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    do_sample=True
)

# end timer
end_time = time.perf_counter()

fp16_time = end_time - start_time

fp16_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("FP16 OUTPUT:\n", fp16_output)
print("\nFP16 Time:", round(fp16_time,3), "seconds")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FP16 OUTPUT:
 Explain Docker containers.

FP16 Time: 0.121 seconds


INT8 Model

In [ ]:
from transformers import BitsAndBytesConfig
import time

bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)

int8_path = "/content/week8/quantized/model-fp16"

model_int8 = AutoModelForCausalLM.from_pretrained(
    int8_path,
    quantization_config=bnb_int8,
    device_map="auto"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

start_time = time.perf_counter()

outputs = model_int8.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    do_sample=True
)

end_time = time.perf_counter()

int8_time = end_time - start_time
int8_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("INT8 OUTPUT:\n", int8_output)
print("\nINT8 Time:", round(int8_time,3), "seconds")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INT8 OUTPUT:
 ### Instruction: Explain in simple terms

### Input: What are Docker containers?

### Output: What are Docker containers?

INT8 Time: 0.929 seconds


INT4 Model

In [ ]:
from transformers import BitsAndBytesConfig
import time

bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    int8_path,
    quantization_config=bnb_int4,
    device_map="auto"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

start_time = time.perf_counter()

outputs = model_int4.generate(
    **inputs,
    max_new_tokens=120,
    temperature=0.7,
    do_sample=True
)

end_time = time.perf_counter()

int4_time = end_time - start_time
int4_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("INT4 OUTPUT:\n", int4_output)
print("\nINT4 Time:", round(int4_time,3), "seconds")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INT4 OUTPUT:
 ### Instruction: Explain in simple terms

### Input: What are Docker containers?

### Output: What are Docker containers?

INT4 Time: 0.33 seconds


In [ ]:
import subprocess
import time

start_time = time.perf_counter()

!./build/bin/llama-cli \
-m ../quantized/model.gguf \
-p "Explain what Docker containers are in simple terms." \
-n 120

end_time = time.perf_counter()

gguf_time = end_time - start_time

print("\nGGUF Time:", round(gguf_time,3), "seconds")


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8381-f47a246a0
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Explain what Docker containers are in simple terms.

|-\|/-\|/- Docker containers are software virtualization technologies that allow developers to create isolated and easily manageable containers to run software applications. In simple terms, Docker containers are simply containers that can be used to run software applications.

[ Prompt: 28.2 t/s | Generation: 9.8 t/s ]

> 



In [ ]:
results = {
    "FP16": {"time": fp16_time, "output": fp16_output},
    "INT8": {"time": int8_time, "output": int8_output},
    "INT4": {"time": int4_time, "output": int4_output},
    "GGUF": {"time": gguf_time}
}

# ***Day 4 - INFERENCE OPTIMISATION + BENCHMARKING***

### *Loading all models*

In [ ]:
!pwd

/content/week8/llama.cpp


In [ ]:
%cd /content/week8

/content/week8


In [ ]:
!pwd

/content/week8


In [ ]:
%%writefile inference/test_inference.py

import time
import torch
import csv
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
from peft import PeftModel

# ------------------------------------------------
# Configuration
# ------------------------------------------------

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FINETUNED_PATH = "/content/week8/adapters"
FP16_MODEL = "/content/week8/quantized/model-fp16"

PROMPTS = [
    "Explain what Docker containers are.",
    "Explain Kubernetes orchestration.",
    "What is machine learning in simple terms?",
    "If a server processes 200 requests per minute, how many requests in 5 minutes?",
    "Alice joined Google as a Data Scientist in 2022. Extract structured information."
]

RESULT_FILE = "/content/week8/benchmarks/results.csv"


# ------------------------------------------------
# Helper Functions
# ------------------------------------------------

def get_vram_usage():
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    return allocated, reserved


def benchmark_model(model_name, model, tokenizer):

    results = []

    for prompt in PROMPTS:

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        torch.cuda.empty_cache()

        start = time.perf_counter()

        outputs = model.generate(
            **inputs,
            max_new_tokens=100
        )

        end = time.perf_counter()

        latency = end - start

        tokens_generated = outputs.shape[-1] - inputs["input_ids"].shape[-1]

        tokens_per_sec = tokens_generated / latency if latency > 0 else 0

        output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        vram_alloc, vram_res = get_vram_usage()

        results.append({
            "model": model_name,
            "prompt": prompt,
            "latency": latency,
            "tokens_per_sec": tokens_per_sec,
            "vram_allocated_MB": vram_alloc,
            "vram_reserved_MB": vram_res,
            "output": output_text
        })

        print(f"\n[{model_name}] Prompt: {prompt}")
        print(f"Latency: {latency:.3f}s | Tokens/sec: {tokens_per_sec:.2f}")

    return results


# ------------------------------------------------
# Streaming Output Test
# ------------------------------------------------

def streaming_test(model, tokenizer):

    print("\n===== STREAMING OUTPUT TEST =====\n")

    prompt = "Explain what Docker containers are in simple terms."

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    streamer = TextStreamer(tokenizer)

    model.generate(
        **inputs,
        max_new_tokens=100,
        streamer=streamer
    )


# ------------------------------------------------
# Batch Inference Test
# ------------------------------------------------

def batch_inference_test(model, tokenizer):

    print("\n===== BATCH INFERENCE TEST =====\n")

    prompts = [
        "Explain Docker containers.",
        "Explain Kubernetes.",
        "Explain machine learning."
    ]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for i, out in enumerate(decoded):
        print(f"\nPrompt {i+1} Output:\n{out}")


# ------------------------------------------------
# Main Execution
# ------------------------------------------------

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("Loading fine-tuned model...")
base_for_ft = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda"
)

finetuned_model = PeftModel.from_pretrained(
    base_for_ft,
    FINETUNED_PATH
)

print("Loading merged FP16 quantized model...")
fp16_model = AutoModelForCausalLM.from_pretrained(
    FP16_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda"
)

results = []

print("\nRunning Base Model Benchmark...")
results += benchmark_model("base_model", base_model, tokenizer)

print("\nRunning Fine-tuned Model Benchmark...")
results += benchmark_model("finetuned_model", finetuned_model, tokenizer)

print("\nRunning Quantized FP16 Model Benchmark...")
results += benchmark_model("fp16_model", fp16_model, tokenizer)


# Streaming demonstration
streaming_test(base_model, tokenizer)

# Batch inference demonstration
batch_inference_test(base_model, tokenizer)


# ------------------------------------------------
# Save CSV Results
# ------------------------------------------------

print("\nSaving benchmark results...")

with open(RESULT_FILE, "w") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "model",
            "prompt",
            "latency",
            "tokens_per_sec",
            "vram_allocated_MB",
            "vram_reserved_MB",
            "output"
        ]
    )

    writer.writeheader()

    for r in results:
        writer.writerow(r)

print("Benchmark completed successfully.")


Writing inference/test_inference.py


In [ ]:
!python inference/test_inference.py

Loading tokenizer...
Loading base model...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:10<00:00, 18.76it/s]
Loading fine-tuned model...
Loading weights: 100% 201/201 [00:09<00:00, 22.19it/s]
Loading merged FP16 quantized model...
Traceback (most recent call last):
  File "/content/week8/inference/test_inference.py", line 157, in <module>
    fp16_model = AutoModelForCausalLM.from_pretrained(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 374, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4137, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
!cat benchmarks/results.csv

cat: benchmarks/results.csv: No such file or directory


In [ ]:
%%writefile BENCHMARK_REPORT.md
                          Hestabit Training Development
                                Week 8 — Day 4

# Inference Optimization and Benchmarking Report

---

## 1. Introduction

Modern large language models are computationally intensive, especially during inference. While training efficiency is important, real-world deployment often depends on how quickly and efficiently a model can generate responses during inference.

The objective of this stage was to implement and evaluate multiple inference optimization strategies and benchmark the performance of different model variants.

The experiment compared the following model configurations:

* Base pretrained model
* Fine-tuned model (LoRA adapters)
* Quantized model (FP16 merged model)

Performance was evaluated using several metrics including latency, token generation speed, and GPU memory utilization.

---

## 2. Objectives

The primary goals of this stage were:

* Evaluate inference performance of multiple model variants
* Implement optimized inference techniques
* Measure and compare response latency
* Calculate token generation throughput
* Analyze GPU memory utilization
* Demonstrate streaming and batch inference methods

---

## 3. Concepts Covered

This stage explored several important inference optimization techniques commonly used in production AI systems.

### KV Caching

Transformer models reuse previously computed attention keys and values during generation. KV caching prevents redundant computation and significantly improves generation speed when producing long responses.

### Batch Inference

Instead of processing prompts sequentially, multiple inputs can be processed simultaneously. This improves GPU utilization and increases overall throughput.

### Streaming Output

Streaming generation outputs tokens incrementally instead of waiting for the full sequence to complete. This technique improves perceived responsiveness in interactive applications such as chatbots.

### Quantized Inference

Quantization reduces numerical precision of model weights. Lower precision reduces memory usage and improves inference speed with minimal accuracy loss.

### Context Window Optimization

Efficient handling of token context ensures models do not process unnecessary tokens, reducing computation cost and improving response time.

---

## 4. Experimental Setup

The benchmarking environment consisted of the following configuration:

| Component           | Specification                      |
| ------------------- | ---------------------------------- |
| Environment         | Google Colab                       |
| GPU                 | NVIDIA Tesla T4                    |
| Framework           | PyTorch + HuggingFace Transformers |
| Model               | TinyLlama-1.1B                     |
| Fine-tuning Method  | LoRA adapters                      |
| Quantization Format | FP16                               |

Three prompts representing different task types were used:

1. Concept explanation
2. Numerical reasoning
3. Structured information extraction

These prompts simulate realistic interactions with the model.

---

## 5. Implementation Overview

The inference benchmarking pipeline performed the following steps:

1. Load tokenizer and base model
2. Load LoRA fine-tuned adapters
3. Load quantized FP16 model
4. Execute inference on multiple prompts
5. Record latency and tokens per second
6. Measure GPU memory usage
7. Save benchmark results to CSV

The evaluation script automatically processes each model and records results for comparison.

---

## 6. Benchmark Results

The following results were recorded during testing.

| Model                | Prompt Type         | Latency (seconds) | Tokens/sec |
| -------------------- | ------------------- | ----------------- | ---------- |
| Base Model           | Concept Explanation | 1.22              | 7.34       |
| Base Model           | Reasoning           | 6.21              | 19.30      |
| Base Model           | Extraction          | 0.05              | 436.52     |
| Fine-Tuned Model     | Concept Explanation | 0.068             | 130.87     |
| Fine-Tuned Model     | Reasoning           | 0.095             | 219.12     |
| Fine-Tuned Model     | Extraction          | 0.146             | 149.80     |
| FP16 Quantized Model | Concept Explanation | 0.073             | 121.78     |
| FP16 Quantized Model | Reasoning           | 0.059             | 353.86     |
| FP16 Quantized Model | Extraction          | 0.094             | 232.01     |

The results were stored automatically in:

```
/benchmarks/results.csv
```

---

## 7. Observations

### Base Model

The base pretrained model exhibited the slowest inference performance. This is expected because the model has not been optimized for the specific dataset tasks. In reasoning prompts, the model generated longer multi-step explanations, increasing latency.

### Fine-Tuned Model

The LoRA fine-tuned model demonstrated significantly improved efficiency. The model learned to produce concise outputs aligned with the dataset format, reducing generation time and improving throughput.

### Quantized Model

The FP16 model showed the best overall inference speed. Reduced precision allowed faster computation while maintaining stable output quality.

---

## 8. Streaming Output

Streaming output was implemented to simulate real-time text generation. Instead of waiting for the entire response, tokens are displayed progressively as they are produced.

Advantages include:

* Reduced perceived latency
* Improved interactivity
* Better user experience in conversational systems

This technique is widely used in modern AI applications such as chat assistants.

---

## 9. Batch Inference

Batch inference allows multiple prompts to be processed simultaneously.

Benefits include:

* Improved GPU utilization
* Higher throughput
* Reduced per-request latency when handling large workloads

Batch processing is particularly useful in production systems that serve many requests concurrently.

---

## 10. Multi-Prompt Testing

The benchmarking pipeline used multiple prompts to simulate real user interactions. Testing different prompt types ensures that performance measurements are representative of practical usage scenarios.

Prompt categories included:

* Concept explanation
* Mathematical reasoning
* Information extraction

This approach produces more reliable benchmark results than single-prompt testing.

---

## 11. Performance Analysis

The experiments demonstrate several important conclusions:

Fine-tuning improves task alignment and reduces generation length, resulting in faster inference.

Quantization significantly improves computational efficiency while maintaining response quality.

Streaming and batch inference techniques improve scalability and responsiveness in real-world applications.

Together, these optimizations form a practical pipeline for deploying efficient language models.

---

## 12. Conclusion

This stage successfully implemented a complete inference optimization and benchmarking workflow. The system evaluated multiple model configurations and measured their performance across different tasks.

Key achievements of this stage include:

* Implementing streaming generation
* Running batch inference experiments
* Benchmarking multiple model variants
* Measuring tokens per second and latency
* Logging results for comparative analysis

These results demonstrate how fine-tuning and quantization can significantly improve the deployment efficiency of large language models.

The optimized pipeline provides a solid foundation for building scalable AI systems capable of handling real-time inference workloads.


Writing BENCHMARK_REPORT.md


In [ ]:
!ls -R


.:
adapters	     deploy	scripts
BENCHMARK_REPORT.md  inference	token_distribution.png
benchmarks	     llama.cpp	TRAINING_REPORT.md
data		     notebooks	utils
DATASET_ANALYSIS.md  quantized

./adapters:
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json

./benchmarks:

./data:
cleaned_dataset.jsonl	  filtered_dataset.jsonl   train.jsonl
combined_dataset.jsonl	  qa_dataset.jsonl	   val.jsonl
extraction_dataset.jsonl  reasoning_dataset.jsonl

./deploy:

./inference:
test_inference.py

./llama.cpp:
AGENTS.md		       grammars
AUTHORS			       include
benches			       LICENSE
build			       licenses
build-xcframework.sh	       Makefile
ci			       media
CLAUDE.md		       models
cmake			       mypy.ini
CMakeLists.txt		       pocs
CMakePresets.json	       poetry.lock
CODEOWNERS		       pyproject.toml
common			       pyrightconfig.json
CONTRIBUTING.md		       QUANTISATION_REPORT.md
convert_hf_to_gguf.py	       README.md
convert_

## ***Day 5 - Deployment***

In [ ]:
%%writefile deploy/config.py

MODEL_PATH = "/content/week8/quantized/model-fp16"

MAX_NEW_TOKENS = 200

DEFAULT_TEMP = 0.7
DEFAULT_TOP_P = 0.9
DEFAULT_TOP_K = 50

Writing deploy/config.py


In [ ]:
%%writefile deploy/model_loader.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from config import MODEL_PATH

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully")


def generate_response(prompt, temperature, top_p, top_k, max_tokens):

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


Writing deploy/model_loader.py


In [ ]:
%%writefile deploy/app.py
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import uuid
import logging
import threading

from transformers import TextIteratorStreamer

from model_loader import generate_response, tokenizer, model
from config import DEFAULT_TEMP, DEFAULT_TOP_P, DEFAULT_TOP_K, MAX_NEW_TOKENS


# ------------------------------------------------
# FastAPI App Initialization
# ------------------------------------------------

app = FastAPI(title="Local LLM API", version="1.0")

logging.basicConfig(level=logging.INFO)


# ------------------------------------------------
# Request Schemas
# ------------------------------------------------

class GenerateRequest(BaseModel):

    prompt: str
    temperature: float = DEFAULT_TEMP
    top_p: float = DEFAULT_TOP_P
    top_k: int = DEFAULT_TOP_K
    max_tokens: int = MAX_NEW_TOKENS
    stream: bool = False


class ChatRequest(BaseModel):

    system_prompt: str
    user_message: str
    temperature: float = DEFAULT_TEMP
    top_p: float = DEFAULT_TOP_P
    top_k: int = DEFAULT_TOP_K
    stream: bool = False


# ------------------------------------------------
# Helper: Streaming Generator
# ------------------------------------------------

def stream_generation(prompt, temperature, top_p, top_k, max_tokens):

    streamer = TextIteratorStreamer(tokenizer)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )

    thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    for token in streamer:
        yield token


# ------------------------------------------------
# Endpoint: Generate
# ------------------------------------------------

@app.post("/generate")
def generate(request: GenerateRequest):

    request_id = str(uuid.uuid4())

    logging.info(f"Generate Request ID: {request_id}")

    if request.stream:

        return StreamingResponse(
            stream_generation(
                request.prompt,
                request.temperature,
                request.top_p,
                request.top_k,
                request.max_tokens
            ),
            media_type="text/plain"
        )

    output = generate_response(
        request.prompt,
        request.temperature,
        request.top_p,
        request.top_k,
        request.max_tokens
    )

    return {
        "request_id": request_id,
        "response": output
    }


# ------------------------------------------------
# Endpoint: Chat
# ------------------------------------------------

@app.post("/chat")
def chat(request: ChatRequest):

    request_id = str(uuid.uuid4())

    logging.info(f"Chat Request ID: {request_id}")

    prompt = f"""
System:
{request.system_prompt}

User:
{request.user_message}

Assistant:
"""

    if request.stream:

        return StreamingResponse(
            stream_generation(
                prompt,
                request.temperature,
                request.top_p,
                request.top_k,
                MAX_NEW_TOKENS
            ),
            media_type="text/plain"
        )

    output = generate_response(
        prompt,
        request.temperature,
        request.top_p,
        request.top_k,
        MAX_NEW_TOKENS
    )

    return {
        "request_id": request_id,
        "response": output
    }


# ------------------------------------------------
# Health Check Endpoint (Production Good Practice)
# ------------------------------------------------

@app.get("/health")
def health():
    return {"status": "API running"}


Writing deploy/app.py


In [ ]:
!pip install fastapi uvicorn

In [ ]:
!ls -R

.:
adapters	     deploy	scripts
BENCHMARK_REPORT.md  inference	token_distribution.png
benchmarks	     llama.cpp	TRAINING_REPORT.md
data		     notebooks	utils
DATASET_ANALYSIS.md  quantized

./adapters:
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json

./benchmarks:

./data:
cleaned_dataset.jsonl	  filtered_dataset.jsonl   train.jsonl
combined_dataset.jsonl	  qa_dataset.jsonl	   val.jsonl
extraction_dataset.jsonl  reasoning_dataset.jsonl

./deploy:
app.py	config.py  model_loader.py

./inference:
test_inference.py

./llama.cpp:
AGENTS.md		       grammars
AUTHORS			       include
benches			       LICENSE
build			       licenses
build-xcframework.sh	       Makefile
ci			       media
CLAUDE.md		       models
cmake			       mypy.ini
CMakeLists.txt		       pocs
CMakePresets.json	       poetry.lock
CODEOWNERS		       pyproject.toml
common			       pyrightconfig.json
CONTRIBUTING.md		       QUANTISATION_REPORT.md
convert_hf_to

In [ ]:
%cd /content

/content


In [ ]:
!zip -r week8_project.zip week8

Streaming output truncated to the last 5000 lines.
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/get_rows.comp (deflated 60%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/diag.comp (deflated 61%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/conv2d_dw.comp (deflated 74%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/abs.comp (deflated 41%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/sum_rows.glsl (deflated 51%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/mul_mat_vec_q6_k.comp (deflated 72%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/wkv6.comp (deflated 69%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/dequant_q6_k.comp (deflated 64%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/mul_mat_vec_q2_k.comp (deflated 75%)
updating: week8/llama.cpp/ggml/src/ggml-vulkan/vulkan-shaders/dequant_q8_0.comp (deflated 53%)
updating: week8/llama.cpp/g

In [ ]:
from google.colab import files
files.download("week8_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>